In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
%pip install catboost

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score
%matplotlib inline

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q3_data.csv")

df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:

df.describe()

In [ ]:
# Task 1: Write your code here:
print("\nMissing Values (df.isnull().sum()):")
print(df.isnull().sum())

df.fillna(df.mean(), inplace=True)

In [ ]:
# Task 2: Write your code here:
def check_duplicates(df):

  #TODO: get duplicated data using pandas
  duplicates = df.duplicated().sum()

  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:
df.info()

# No need for encoding

In [ ]:
# Task 4: Write your code here:

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
num_features = df.select_dtypes(include=['float64', 'int64']).columns.drop("Target")
df[num_features] = scaler.fit_transform(df[num_features])
df

In [ ]:
# Task 5: Write your code here:

target_counts = df['Target'].value_counts()
target_counts

# our target is imbalanced

In [ ]:
# Task 1: Write your code here:

X = df.drop("Target", axis=1)
y = df['Target']

In [ ]:
# Task 2,3,4,5: Write your code here:

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1_scores = []

for train_index, test_index in skf.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model = CatBoostClassifier(verbose=0)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    f1_scores.append(f1_score(y_test, y_pred))

print(f"Average F1 Score: {sum(f1_scores) / len(f1_scores)}")



In [ ]:
# Task 1: Write your code here:
feature_importances = model.get_feature_importance()
features = X.columns

plt.figure(figsize=(10, 6))
plt.barh(features, feature_importances)
plt.xlabel("Feature Importance")
plt.title("Feature Importance from CatBoostClassifier")
plt.show()


In [ ]:
# Task 2: Write your code here:
golden_feature = features[feature_importances.argmax()]
print(f"The golden feature is: {golden_feature}")

In [ ]:
# Task Bonus: Write your code here:

X_golden = X[[golden_feature]]

f1_scores_golden = []

for train_index, test_index in skf.split(X_golden, y):
    X_train, X_test = X_golden.iloc[train_index], X_golden.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model_golden = CatBoostClassifier(verbose=0)
    model_golden.fit(X_train, y_train)

    y_pred_golden = model_golden.predict(X_test)
    f1_scores_golden.append(f1_score(y_test, y_pred_golden))

print(f"Average F1 Score with only golden feature: {sum(f1_scores_golden) / len(f1_scores_golden)}")
print(f"Compared to the full model: {sum(f1_scores) / len(f1_scores)}")